# Kraken — End-to-end exploration

This notebook walks through the full pipeline on a single processed SAR scene:
1. Load a preprocessed Sentinel-1 GeoTIFF
2. Run CFAR detection
3. Run YOLO detection (if weights available)
4. Visualise overview and detection crops
5. Inspect combined GeoJSON


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# project root on path
ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(ROOT))

from config.settings import OUTPUTS_DIR, PROCESSED_DIR, MODELS_DIR
print(f"OUTPUTS_DIR : {OUTPUTS_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

## 1 · Pick a scene

Point `SCENE_PATH` at any `*_processed.tif` you have on disk.

In [ ]:
# ── configure ────────────────────────────────────────────────────────────────
scenes = sorted(PROCESSED_DIR.glob("*_processed.tif"))
print(f"Found {len(scenes)} processed scene(s):")
for s in scenes:
    print(" ", s.name)

# Pick the first one (or set manually)
SCENE_PATH = scenes[0] if scenes else None
YOLO_WEIGHTS = MODELS_DIR / "yolo_vessel" / "weights" / "best.pt"
OUT_DIR = OUTPUTS_DIR / "notebook"

print(f"\nUsing: {SCENE_PATH}")

## 2 · Peek at raw bands

In [ ]:
import rasterio
from src.utils.viz import sar_to_rgb

with rasterio.open(SCENE_PATH) as src:
    # Read at 1/4 resolution for display speed
    ds = 4
    out_shape = (src.count, src.height // ds, src.width // ds)
    data_small = src.read(out_shape=out_shape, resampling=rasterio.enums.Resampling.average)
    full_transform = src.transform
    crs = src.crs

print(f"Full scene shape (bands, H, W): {data_small.shape[0]}, {data_small.shape[1]*ds}, {data_small.shape[2]*ds}")
print(f"CRS: {crs}")
print(f"Transform: {full_transform}")

rgb = sar_to_rgb(data_small)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(rgb)
axes[0].set_title("VV (R), VH (G), VV (B)")
axes[0].axis("off")
axes[1].imshow(data_small[0], cmap="gray", vmin=np.nanpercentile(data_small[0], 2),
               vmax=np.nanpercentile(data_small[0], 98))
axes[1].set_title("VV σ⁰ (dB)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3 · CFAR detection

In [ ]:
from src.detect.cfar_detect import detect_scene

cfar_boxes = detect_scene(
    SCENE_PATH,
    out_dir=OUT_DIR,
)
print(f"\nCFAR found {len(cfar_boxes)} vessel candidates")

## 4 · YOLO detection (skip if no weights)

In [ ]:
yolo_boxes = []

if YOLO_WEIGHTS.exists():
    from src.detect.yolo_detect import run_yolo_on_scene
    yolo_boxes = run_yolo_on_scene(SCENE_PATH, YOLO_WEIGHTS, out_dir=OUT_DIR)
    print(f"YOLO found {len(yolo_boxes)} vessel candidates")
else:
    print(f"[SKIP] YOLO weights not found at {YOLO_WEIGHTS}")
    print("       Train first:  python -m src.train.train_yolo --help")

## 5 · Scene overview

In [ ]:
from src.utils.viz import render_scene_overview

render_scene_overview(
    SCENE_PATH,
    cfar_boxes=cfar_boxes or None,
    yolo_boxes=yolo_boxes or None,
    out_path=OUT_DIR / f"{SCENE_PATH.stem}_overview.png",
)

## 6 · Detection chip grid

In [ ]:
from src.utils.viz import render_detection_grid

all_boxes = cfar_boxes + yolo_boxes
if all_boxes:
    render_detection_grid(
        SCENE_PATH,
        all_boxes,
        out_path=OUT_DIR / f"{SCENE_PATH.stem}_chips.png",
        max_chips=48,
    )
else:
    print("No detections to show.")

## 7 · Combined GeoJSON

In [ ]:
import json
from src.utils.geo_utils import detections_to_geojson, merge_geojson, save_geojson

stem = SCENE_PATH.stem
cfar_fc = detections_to_geojson(cfar_boxes, stem, "cfar")
yolo_fc = detections_to_geojson(yolo_boxes, stem, "yolo")
combined = merge_geojson(cfar_fc, yolo_fc)

out_geojson = OUT_DIR / f"{stem}_detections.geojson"
save_geojson(combined, out_geojson)

print(f"\nTotal features: {len(combined['features'])}")
if combined['features']:
    print("First feature:")
    print(json.dumps(combined['features'][0], indent=2))

## 8 · Quick stats

In [ ]:
from collections import Counter

by_detector = Counter(f["properties"]["detector"] for f in combined["features"])
by_class    = Counter(f["properties"]["cls_name"] for f in combined["features"])

print("Detections by detector:", dict(by_detector))
print("Detections by class:   ", dict(by_class))

confs = [f["properties"]["conf"]
         for f in combined["features"]
         if "conf" in f["properties"]]
if confs:
    print(f"Confidence  min={min(confs):.3f}  mean={np.mean(confs):.3f}  max={max(confs):.3f}")